In [1]:
import joblib
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score
import warnings
from sklearn.exceptions import UndefinedMetricWarning, InconsistentVersionWarning

from inference_monitor import measure_inference

In [12]:

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

In [13]:
names = ["decision_tree","k-nn", "random_forest", "logistic_regression","naive_bayes", "svm_linear", "xgboost"]

In [14]:
# import joblib
# import pandas as pd
# import os
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"

# df = pd.read_csv("../../data/processed/processed_Evil_Twin_one_line.csv")
# X = df.drop(columns=["Label"])
# y = df["Label"]

# one_line_results = []

# for i in range(len(names)):
#     y_pred,measures = measure_inference(joblib.load(f"../models/{names[i]}.joblib").predict, X)
#     measures["Model"] = names[i]
#     one_line_results.append(measures)
# results_df = pd.DataFrame(one_line_results, columns=["Model", "Time (s)", "Avg CPU (%)", "Max CPU (%)", "Pred RAM (MB)"])

# results_df.to_csv("../results/inference_benchmarks.csv", index=False)

In [4]:
df = pd.read_csv("../results/model_evaluation_results-41-51.csv")
df

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
0,decision_tree,0.889209,0.886046,0.889209,0.843757,[[400058 35]\n [ 49821 83]]
1,k-nn,0.997818,0.997880,0.997818,0.997799,[[399701 392]\n [ 590 49314]]
2,random_forest,0.961726,0.986988,0.961726,0.972206,[[383277 16816]\n [ 407 49497]]
3,logistic_regression,0.973487,0.975840,0.973487,0.971512,[[400073 20]\n [ 11911 37993]]
4,naive_bayes,0.817946,0.738701,0.817946,0.767321,[[359083 41010]\n [ 40914 8990]]
5,svm_linear,0.972436,0.956197,0.972436,0.963204,[[399956 137]\n [ 12267 37637]]
6,xgboost,0.999109,0.999110,0.999109,0.999007,[[400031 62]\n [ 339 49565]]


In [6]:
pd.read_csv("../results/inference_benchmarks.csv")

,Model,Time (s),Avg CPU (%),Max CPU (%),Pred RAM (MB)
0,decision_tree,0.0029,148.60,148.6,0.1367
1,k-nn,0.1101,94.93,153.1,1.8164
2,random_forest,0.0938,85.81,307.5,1.2617
3,logistic_regression,0.0028,148.40,148.4,0.0391
4,naive_bayes,0.0027,0.00,0.0,0.0195
5,svm_linear,0.0030,0.00,0.0,0.0742
6,xgboost,0.0138,66.95,133.9,0.8125


In [15]:
results = []

In [16]:
for name in names:
    model = joblib.load(f"../models/{name}.joblib")
    j=0
    acc_sum = 0
    prec_sum = 0
    rec_sum = 0
    f1_sum = 0
    cm_sum = [[0,0],[0,0]]
    print(f"Testing model: {name}")

    for i in range(51,61):
        if os.path.exists(f"../../data/processed_raw/processed_Evil_Twin_{i}.csv"):
            print(f"Testing on dataset: processed_Evil_Twin_{i}.csv")
            j+=1
            df = pd.read_csv(f"../../data/processed_raw/processed_Evil_Twin_{i}.csv")
            X = df.drop("Label", axis=1)
            y = df["Label"]
            y_pred = model.predict(X)
            acc = accuracy_score(y, y_pred)
            cr = classification_report(y, y_pred,output_dict=True)
            acc_sum += acc
            prec_sum += cr["weighted avg"]["precision"]
            rec_sum += cr["weighted avg"]["recall"]
            f1_sum += cr["weighted avg"]["f1-score"]
            cm_sum += confusion_matrix(y, y_pred)

    results.append({"Model": name, "Accuracy": acc_sum/j, "Precision": prec_sum/j, "Recall": rec_sum/j, "F1-Score": f1_sum/j, "Confusion Matrix": cm_sum})

Testing model: decision_tree
Testing on dataset: processed_Evil_Twin_51.csv
Testing on dataset: processed_Evil_Twin_52.csv
Testing on dataset: processed_Evil_Twin_53.csv
Testing on dataset: processed_Evil_Twin_54.csv
Testing on dataset: processed_Evil_Twin_56.csv
Testing on dataset: processed_Evil_Twin_57.csv
Testing on dataset: processed_Evil_Twin_58.csv
Testing on dataset: processed_Evil_Twin_59.csv
Testing on dataset: processed_Evil_Twin_60.csv
Testing model: k-nn
Testing on dataset: processed_Evil_Twin_51.csv
Testing on dataset: processed_Evil_Twin_52.csv
Testing on dataset: processed_Evil_Twin_53.csv
Testing on dataset: processed_Evil_Twin_54.csv
Testing on dataset: processed_Evil_Twin_56.csv
Testing on dataset: processed_Evil_Twin_57.csv
Testing on dataset: processed_Evil_Twin_58.csv
Testing on dataset: processed_Evil_Twin_59.csv
Testing on dataset: processed_Evil_Twin_60.csv
Testing model: random_forest
Testing on dataset: processed_Evil_Twin_51.csv
Testing on dataset: processed

In [17]:
df_results = pd.DataFrame(results).set_index("Model")
df_results

,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
Model,,,,,
decision_tree,0.883473,0.958143,0.883473,0.907034,"[[397113, 42183], [10254, 464]]"
k-nn,0.998867,0.999244,0.998867,0.999024,"[[438875, 421], [89, 10629]]"
random_forest,0.998380,0.999252,0.998380,0.998732,"[[438570, 726], [3, 10715]]"
logistic_regression,0.993700,0.994558,0.993700,0.993599,"[[438876, 420], [2415, 8303]]"
naive_bayes,0.976182,0.956748,0.976182,0.965490,"[[439296, 0], [10718, 0]]"
svm_linear,0.996811,0.998702,0.996811,0.997533,"[[438242, 1054], [381, 10337]]"
xgboost,0.995782,0.996164,0.995782,0.995586,"[[439139, 157], [1741, 8977]]"


In [26]:
df_results.sort_values(by="F1-Score", ascending=False)

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
1,k-nn,0.998867,0.999244,0.998867,0.999024,[[438875 421]\n [ 89 10629]]
2,random_forest,0.998380,0.999252,0.998380,0.998732,[[438570 726]\n [ 3 10715]]
5,svm_linear,0.996811,0.998702,0.996811,0.997533,[[438242 1054]\n [ 381 10337]]
6,xgboost,0.995782,0.996164,0.995782,0.995586,[[439139 157]\n [ 1741 8977]]
3,logistic_regression,0.993700,0.994558,0.993700,0.993599,[[438876 420]\n [ 2415 8303]]
4,naive_bayes,0.976182,0.956748,0.976182,0.965490,[[439296 0]\n [ 10718 0]]
0,decision_tree,0.883473,0.958143,0.883473,0.907034,[[397113 42183]\n [ 10254 464]]


In [18]:
df_results.to_csv("../results/model_evaluation_results-51-61.csv")

In [19]:
df_results = pd.read_csv("../results/model_evaluation_results-51-61.csv")
df_results

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
0,decision_tree,0.883473,0.958143,0.883473,0.907034,[[397113 42183]\n [ 10254 464]]
1,k-nn,0.998867,0.999244,0.998867,0.999024,[[438875 421]\n [ 89 10629]]
2,random_forest,0.998380,0.999252,0.998380,0.998732,[[438570 726]\n [ 3 10715]]
3,logistic_regression,0.993700,0.994558,0.993700,0.993599,[[438876 420]\n [ 2415 8303]]
4,naive_bayes,0.976182,0.956748,0.976182,0.965490,[[439296 0]\n [ 10718 0]]
5,svm_linear,0.996811,0.998702,0.996811,0.997533,[[438242 1054]\n [ 381 10337]]
6,xgboost,0.995782,0.996164,0.995782,0.995586,[[439139 157]\n [ 1741 8977]]


In [20]:
df_results = pd.read_csv("../results/model_evaluation_results-41-51.csv")
df_results

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
0,decision_tree,0.889209,0.886046,0.889209,0.843757,[[400058 35]\n [ 49821 83]]
1,k-nn,0.997818,0.997880,0.997818,0.997799,[[399701 392]\n [ 590 49314]]
2,random_forest,0.961726,0.986988,0.961726,0.972206,[[383277 16816]\n [ 407 49497]]
3,logistic_regression,0.973487,0.975840,0.973487,0.971512,[[400073 20]\n [ 11911 37993]]
4,naive_bayes,0.817946,0.738701,0.817946,0.767321,[[359083 41010]\n [ 40914 8990]]
5,svm_linear,0.972436,0.956197,0.972436,0.963204,[[399956 137]\n [ 12267 37637]]
6,xgboost,0.999109,0.999110,0.999109,0.999007,[[400031 62]\n [ 339 49565]]


In [21]:
df_results = pd.read_csv("../results/model_evaluation_results-51-61.csv")
df_results

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
0,decision_tree,0.883473,0.958143,0.883473,0.907034,[[397113 42183]\n [ 10254 464]]
1,k-nn,0.998867,0.999244,0.998867,0.999024,[[438875 421]\n [ 89 10629]]
2,random_forest,0.998380,0.999252,0.998380,0.998732,[[438570 726]\n [ 3 10715]]
3,logistic_regression,0.993700,0.994558,0.993700,0.993599,[[438876 420]\n [ 2415 8303]]
4,naive_bayes,0.976182,0.956748,0.976182,0.965490,[[439296 0]\n [ 10718 0]]
5,svm_linear,0.996811,0.998702,0.996811,0.997533,[[438242 1054]\n [ 381 10337]]
6,xgboost,0.995782,0.996164,0.995782,0.995586,[[439139 157]\n [ 1741 8977]]


In [22]:
df_results.sort_values("F1-Score", ascending=False)

,Model,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
1,k-nn,0.998867,0.999244,0.998867,0.999024,[[438875 421]\n [ 89 10629]]
2,random_forest,0.998380,0.999252,0.998380,0.998732,[[438570 726]\n [ 3 10715]]
5,svm_linear,0.996811,0.998702,0.996811,0.997533,[[438242 1054]\n [ 381 10337]]
6,xgboost,0.995782,0.996164,0.995782,0.995586,[[439139 157]\n [ 1741 8977]]
3,logistic_regression,0.993700,0.994558,0.993700,0.993599,[[438876 420]\n [ 2415 8303]]
4,naive_bayes,0.976182,0.956748,0.976182,0.965490,[[439296 0]\n [ 10718 0]]
0,decision_tree,0.883473,0.958143,0.883473,0.907034,[[397113 42183]\n [ 10254 464]]
